<a href="https://colab.research.google.com/github/3iqpotato/softuni_course_project/blob/main/softuni_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai pypdf chromadb elevenlabs langchain-text-splitters

#Imports

In [ ]:
from enum import Enum
from google.colab import userdata
import json

from openai import OpenAI
from pydantic import BaseModel, Field

import chromadb
from pypdf import PdfReader


from elevenlabs import ElevenLabs

from IPython.display import display, Audio, Image
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

#Requirements and constants

In [ ]:
SECRET_NAMES = {
    "OPENAI_API_KEY": "Open_AI_Free_Key",
    "ELEVENLABS_API_KEY": "Eleven_labs_api_key",
}


MODELS = {
    "Embedding_Model": "text-embedding-3-small",
    "Gpt_4o_model": "gpt-4o-mini"

}

#Helping Functions

In [ ]:
def load_api_keys(secret_names: dict) -> dict:

    keys = {}
    for internal_name, secret_name in secret_names.items():
        try:
            value = userdata.get(secret_name)
        except userdata.SecretNotFoundError:
            raise EnvironmentError(
                f"Secret '{secret_name}' was not found in Colab Secrets. "
                f"Make sure the name matches exactly (the key icon on the left)."
            )
        except userdata.NotebookAccessError:
            raise EnvironmentError(
                f"Secret '{secret_name}' exists, but the notebook does not have access to it. "
                f"Enable 'Notebook access' in the Colab Secrets panel."
            )

        if not value:
            raise EnvironmentError(f"Secret '{secret_name}' is empty.")

        keys[internal_name] = value

    return keys


API_KEYS = load_api_keys(SECRET_NAMES)



#TOOLS

In [ ]:
def ingest_pdf(pdf_path: str, chunk_size: int = 800, chunk_overlap: int = 100) -> int:
    """
    Reads a PDF, splits it into chunks, and stores them in the Chroma collection.
    Returns the number of stored chunks.
    """
    reader = PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)

    if not full_text.strip():
        raise ValueError(f"Could not extract text from '{pdf_path}' — it may be a scanned PDF without a text layer.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_text(full_text)

    ids = [f"chunk_{i}" for i in range(len(chunks))]
    collection.add(documents=chunks, ids=ids)

    print(f"Stored {len(chunks)} chunks from '{pdf_path}' in the collection.")
    return len(chunks)


def retrieve_information(prompt: str) -> str:
    """
    Searches the Chroma collection for the chunks most relevant to the prompt
    and returns an answer generated by the LLM based on them.
    """
    results = collection.query(query_texts=[prompt], n_results=4)
    retrieved_chunks = results["documents"][0] if results["documents"] else []

    if not retrieved_chunks:
        return "I could not find relevant information in the document for this question."

    context = "\n\n---\n\n".join(retrieved_chunks)

    completion = client.chat.completions.create(
        model=MODELS["Gpt_4o_model"],
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question ONLY based on the provided document context. "
                    "If the answer is not contained in the context, say that the information is missing."
                ),
            },
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {prompt}"},
        ],
    )

    return completion.choices[0].message.content

#TOOL DEFINITIONS

In [ ]:
retrieve_information_tool = {
    "type": "function",
    "function": {
        "name": "retrieve_information",
        "description": (
            "Searches the uploaded PDF document (using a vector database with embeddings) "
            "and returns a text answer to a specific question about the document. "
            "Use this function whenever the user requests information related to the document's content."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "The question to ask about the document.",
                }
            },
            "required": ["prompt"],
            "additionalProperties": False,
        },
    },
}


TOOLS = [
    retrieve_information_tool,
]


AVAILABLE_FUNCTIONS = {
    "retrieve_information": retrieve_information,
}

#STRUCTURED OUTPUT SCHEMAS


In [ ]:
class ResponseFormat(str, Enum):
    text = "text"
    image = "image"
    audio = "audio"


class AskAIRequest(BaseModel):
    """
    Schema for OpenAI's structured response — extracts from the user's free-form
    question the actual prompt for the document
    and the format in which the user wants to receive the answer.
    """
    prompt: str = Field(
        description=(
            "The actual question for the document, stripped of any formatting instructions "
            "(e.g. 'as an image', 'read aloud', etc.) — only the core question."
        )
    )
    format: ResponseFormat = Field(
        description=(
            "The user's preferred response format: "
            "'text' if they want a text response or did not specify a format, "
            "'image' if they want the answer displayed as an image, "
            "'audio' if they want the answer read aloud."
        )
    )

#Load Clients

In [ ]:
client = OpenAI(api_key=API_KEYS["OPENAI_API_KEY"])
elevenlabs_client = ElevenLabs(api_key=API_KEYS["ELEVENLABS_API_KEY"])

#Chromma client and collection
chroma_client = chromadb.Client()

openai_ef = OpenAIEmbeddingFunction(
    api_key=API_KEYS["OPENAI_API_KEY"],
    model_name=MODELS["Embedding_Model"],
)
collection = chroma_client.get_or_create_collection(
    name="document_chunks",
    embedding_function=openai_ef,
)